# Recommendation
Here we have dataset of users with their rating for each item. **If user has not rated an item, then it is marked as 0**.
We have to find a good recommendation system that would recommend items to user that he has not rated: In essence it would predict the rating the user is likely to give to an un-rated items. If the predicted ratings for unrated items are high then those can be recommended to the user.

Assume itemA, itemB, itemC belong to one class, say kitchenware(or in case of movies an ACTION genre), and itemD, itemE, itemF belong to another class, say bathware(or in case of movies a COMEDY genre)

- This can be used to recommend products like movies, shopping items, etc to customers

In [59]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.linalg import svd

# Sample user-item rating matrix (users as rows, items as columns)
ratings = np.array([
    [5, 5, 4, 1, 1, 1], # user1 ratings 
    [4, 5, 5, 0, 1, 1], # user2 ratings
    [5, 4, 5, 1, 0, 0],
    [4, 5, 4, 1, 1, 0],
    [5, 5, 5, 1, 1, 1],
    [0, 1, 1, 4, 5, 0],
    [1, 2, 1, 5, 0, 5],
    [1, 2, 1, 5, 5, 4],
    [1, 0, 1, 4, 5, 0],
    
], dtype=float)
Items = ["ItemA", "ItemB", "ItemC", "ItemD", "ItemE", "ItemF"]
Users = ["User1", "User2", "User3", "User4", "User5", "User6", "User7", "User8", "User9"]
df = pd.DataFrame(ratings, columns=Items,
                             index=Users)
print("Original Rating Matrix:\n", df)

# file = "recommend_rated_item.csv"
# df.to_csv(file)
# df = pd.read_csv(file)
# df = df.drop(columns=['Unnamed: 0'])
# Items = df.columns.tolist()
# Users = df.index.tolist()

Original Rating Matrix:
        ItemA  ItemB  ItemC  ItemD  ItemE  ItemF
User1    5.0    5.0    4.0    1.0    1.0    1.0
User2    4.0    5.0    5.0    0.0    1.0    1.0
User3    5.0    4.0    5.0    1.0    0.0    0.0
User4    4.0    5.0    4.0    1.0    1.0    0.0
User5    5.0    5.0    5.0    1.0    1.0    1.0
User6    0.0    1.0    1.0    4.0    5.0    0.0
User7    1.0    2.0    1.0    5.0    0.0    5.0
User8    1.0    2.0    1.0    5.0    5.0    4.0
User9    1.0    0.0    1.0    4.0    5.0    0.0


Here we see that users who have given high rating to item A to C (ACTION genre) have  given low rating to item D to F (COMEDY genre) and vice versa.
Few of the users have given no ratings, as depicted by entry 0. In reality this matrix would be sparse, meaning most the entries would be 0 and only handful would be ono-zero: Users only rate handfull of movies.

In [62]:
# Apply SVD
U, S, VT = svd(ratings, full_matrices=False)
print(f"Matrix U: \n{np.round(U,2)}\n\n")
# print(f"Sigma diagonal-singularvalues: \n{np.round(S,2)}\n\n")
print(f"Sigma matrix: \n{np.diag(np.round(S,2))}\n\n")
print(f"Matrix VT: \n{np.round(VT,2)}\n\n")

Matrix U: 
[[-0.41  0.16  0.01 -0.08  0.17 -0.64]
 [-0.4   0.21 -0.03 -0.44 -0.03  0.61]
 [-0.39  0.24 -0.05  0.57  0.18  0.24]
 [-0.38  0.16 -0.11 -0.04 -0.56 -0.35]
 [-0.44  0.18 -0.01 -0.01  0.18  0.09]
 [-0.17 -0.44 -0.41  0.04 -0.56  0.14]
 [-0.23 -0.33  0.78  0.32 -0.23  0.08]
 [-0.28 -0.56  0.13 -0.48  0.33 -0.08]
 [-0.17 -0.44 -0.43  0.38  0.34 -0.01]]


Sigma matrix: 
[[19.61  0.    0.    0.    0.    0.  ]
 [ 0.   11.31  0.    0.    0.    0.  ]
 [ 0.    0.    5.93  0.    0.    0.  ]
 [ 0.    0.    0.    2.13  0.    0.  ]
 [ 0.    0.    0.    0.    1.32  0.  ]
 [ 0.    0.    0.    0.    0.    1.09]]


Matrix VT: 
[[-0.51 -0.55 -0.52 -0.28 -0.24 -0.18]
 [ 0.27  0.2   0.24 -0.64 -0.57 -0.3 ]
 [-0.05  0.1  -0.12  0.18 -0.62  0.74]
 [ 0.3  -0.42  0.15  0.61 -0.42 -0.4 ]
 [ 0.59 -0.6   0.01 -0.31  0.22  0.38]
 [-0.48 -0.32  0.8  -0.08 -0.    0.16]]




Here we decompose A:

A = U\*S\*VT

We will work with only first 2 sigma values, because they have large magnitude (19.61 and 11.31) and thus capture the most information.
<img src="images/A_decompose.png"  width="800" height="400">

In [63]:
# Keep top k singular values (latent features)
k = 2  # dimensionality reduction
U_k = np.round(U[:, :k], 2)
print(f"Matrix U_k: \n{U_k}\n\n")

S_k = np.round(np.diag(S[:k]), 2)
print(f"Sigma_k matrix: \n{S_k}\n\n")

VT_k = np.round(VT[:k, :], 2)
print(f"Matrix VT_k: \n{VT_k}\n\n")

# Reconstruct the matrix with reduced dimensions
reconstructed = np.dot(U_k, np.dot(S_k, VT_k))
predicted_ratings = pd.DataFrame(reconstructed,
                                 columns=Items, # these are optional
                                 index=Users # these are optional
                                )
print("\nPredicted Ratings (via SVD):\n", predicted_ratings.round(2))

Matrix U_k: 
[[-0.41  0.16]
 [-0.4   0.21]
 [-0.39  0.24]
 [-0.38  0.16]
 [-0.44  0.18]
 [-0.17 -0.44]
 [-0.23 -0.33]
 [-0.28 -0.56]
 [-0.17 -0.44]]


Sigma_k matrix: 
[[19.61  0.  ]
 [ 0.   11.31]]


Matrix VT_k: 
[[-0.51 -0.55 -0.52 -0.28 -0.24 -0.18]
 [ 0.27  0.2   0.24 -0.64 -0.57 -0.3 ]]



Predicted Ratings (via SVD):
        ItemA  ItemB  ItemC  ItemD  ItemE  ItemF
User1   4.59   4.78   4.62   1.09   0.90   0.90
User2   4.64   4.79   4.65   0.68   0.53   0.70
User3   4.63   4.75   4.63   0.40   0.29   0.56
User4   4.29   4.46   4.31   0.93   0.76   0.80
User5   4.95   5.15   4.98   1.11   0.91   0.94
User6   0.36   0.84   0.54   4.12   3.64   2.09
User7   1.29   1.73   1.45   3.65   3.21   1.93
User8   1.09   1.75   1.34   5.59   4.93   2.89
User9   0.36   0.84   0.54   4.12   3.64   2.09


## observation
Here we compare the ratings of the user with his own ratings for all items. For e.g. user2 ratings for items are 4.64, 4.79, 4.65, 0.68, 0.53, 0.70 He did not rated itemD and we can see that he would likely give it low rating compared with ACTION genre: itemA, itemB, itemC. So do not recommend him itemD

<img src="images/rating_predict_analyse.png"  width="400" height="200">

- For user2 who did not rate itemD, he would rate it at low end. So do not recommend him itemD
- For user3 who did not rate itemE and itemF, he would rate these at low end. So do not recommend him itemE and itemF
- For user6 who did not rate itemF, he would rate it at high end. So do recommend him itemF
- For user7 who did not rate itemE, he would rate it at high end. So do recommend him itemE
- For user9 who did not rate itemF, he would rate it at high end. So do recommend him itemF